# Olist 电商 KPI 指标分析

基于 `fact_order` 宽表，计算销售、客户、物流、支付四大维度共 10 项核心指标。

In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 连接数据库
DB_PATH = "olist.db"
conn = sqlite3.connect(DB_PATH)

# 加载宽表
df = pd.read_sql("SELECT * FROM fact_order", conn)
# 时间字段转换
df['purchase_time']  = pd.to_datetime(df['purchase_time'])
df['first_purchase_date'] = pd.to_datetime(df['first_purchase_date'])
df['last_purchase_date']  = pd.to_datetime(df['last_purchase_date'])

print(f"加载完成：{len(df):,} 行 × {len(df.columns)} 列")
print(f"订单数: {df['order_id'].nunique():,}")
print(f"客户数: {df['customer_id'].nunique():,}")
print(f"时间范围: {df['purchase_time'].min().date()} ~ {df['purchase_time'].max().date()}")
df.head(3)

加载完成：112,650 行 × 38 列
订单数: 98,666
客户数: 98,666
时间范围: 2016-09-04 ~ 2018-09-03


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,purchase_time,logistics_days,...,seller_city,seller_state,customer_unique_id,customer_city,customer_state,first_purchase_date,last_purchase_date,purchase_frequency,total_spent,avg_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,7.0,...,volta redonda,SP,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,2017-09-13 08:59:02,2017-09-13 08:59:02,1,72.19,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,16.0,...,sao paulo,SP,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,2017-04-26 10:53:06,2017-04-26 10:53:06,1,259.83,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,7.0,...,borda da mata,MG,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,2018-01-14 14:33:31,2018-01-14 14:33:31,1,216.87,216.87


---
## 一、销售指标

In [2]:
# ==== 1. 总销售额 GMV、总订单量、总商品销量 ====
total_gmv   = df['order_total_amount'].sum()
total_orders = df['order_id'].nunique()
total_items  = df['order_item_count'].sum()

print("【指标 1 — 整体概览】")
print(f"  总销售额 (GMV) : R$ {total_gmv:>15,.2f}")
print(f"  总订单量       : {total_orders:>15,}")
print(f"  总商品销量     : {total_items:>15,.0f}")

# ==== 2. 平均客单价、平均订单商品数 ====
# 每笔订单一行（去重 order_id）
order_level = df.groupby('order_id').agg(
    order_amount = ('order_total_amount', 'first'),
    item_count   = ('order_item_count',    'first')
).reset_index()

avg_order_value = order_level['order_amount'].mean()
avg_item_count  = order_level['item_count'].mean()

print("\n【指标 2 — 客单价与商品数】")
print(f"  平均客单价 (AOV) : R$ {avg_order_value:>12,.2f}")
print(f"  平均订单商品数   : {avg_item_count:>12.1f} 件")

print("\n📊 **业务解读**：")
print("  总 GMV 反映平台整体交易规模。")
print("  平均客单价和商品数反映用户单次购买力度，客单价高说明高价值商品占比大。")
print("  若平均商品数接近 1，说明用户以单件购买为主，适合做推荐与凑单促销。")

【指标 1 — 整体概览】
  总销售额 (GMV) : R$   20,304,557.35
  总订单量       :          98,666
  总商品销量     :         157,222

【指标 2 — 客单价与商品数】
  平均客单价 (AOV) : R$       160.58
  平均订单商品数   :          1.1 件

📊 **业务解读**：
  总 GMV 反映平台整体交易规模。
  平均客单价和商品数反映用户单次购买力度，客单价高说明高价值商品占比大。
  若平均商品数接近 1，说明用户以单件购买为主，适合做推荐与凑单促销。


In [3]:
# ==== 3. 月度/季度销售趋势 ====

# 月度趋势
monthly = df.groupby('purchase_ym').agg(
    monthly_gmv    = ('order_total_amount', 'sum'),
    monthly_orders = ('order_id',           'nunique'),
    monthly_items  = ('order_item_count',   'sum')
).reset_index()
monthly.columns = ['月份', '销售额', '订单量', '商品数']

# 季度趋势
quarterly = df.groupby('purchase_quarter').agg(
    q_gmv    = ('order_total_amount', 'sum'),
    q_orders = ('order_id',           'nunique')
).reset_index()
quarterly.columns = ['季度', '销售额', '订单量']

print("【指标 3 — 月度销售趋势】")
print(monthly.to_string(index=False))

print("\n【指标 3 — 季度汇总】")
print(quarterly.to_string(index=False))

# 计算环比增长率
if len(monthly) >= 2:
    latest = monthly.iloc[-1]
    prev   = monthly.iloc[-2]
    gmv_growth = (latest['销售额'] - prev['销售额']) / prev['销售额'] * 100
    print(f"\n  最后一月环比增长: {gmv_growth:+.1f}%")

print("\n📊 **业务解读**：")
print("  月度趋势揭示增长节奏：持续上升说明处于成长期，")
print("  月度波动大则可能与促销活动或季节性因素有关。")

【指标 3 — 月度销售趋势】
     月份        销售额  订单量   商品数
2016-09     777.90    3    14
2016-10   73308.86  308   551
2016-12      19.62    1     1
2017-01  187702.56  789  1681
2017-02  344074.59 1733  2553
2017-03  526923.26 2641  4030
2017-04  505513.70 2391  3504
2017-05  724271.99 3660  5468
2017-06  600578.09 3217  4605
2017-07  737158.80 3969  6421
2017-08  869923.44 4293  6984
2017-09 1015731.60 4243  6653
2017-10 1021140.06 4568  7906
2017-11 1583583.12 7451 12595
2017-12 1042728.24 5624  8394
2018-01 1407333.20 7220 11288
2018-02 1305645.51 6694 11138
2018-03 1475478.96 7188 11457
2018-04 1496756.38 6934 11471
2018-05 1506912.96 6853 11273
2018-06 1297482.46 6160  9912
2018-07 1351608.49 6273  9680
2018-08 1229737.10 6452  9642
2018-09     166.46    1     1

【指标 3 — 季度汇总】
 季度        销售额   订单量
  1 5247158.08 26265
  2 6131515.58 29215
  3 5205103.79 25234
  4 3720779.90 17952

  最后一月环比增长: -100.0%

📊 **业务解读**：
  月度趋势揭示增长节奏：持续上升说明处于成长期，
  月度波动大则可能与促销活动或季节性因素有关。


In [4]:
# ==== 4. 各产品分类销售额排行 Top 10 ====
category_sales = df.groupby('product_category_name_english').agg(
    sales   = ('order_total_amount', 'sum'),
    orders  = ('order_id',           'nunique'),
    items   = ('order_item_count',   'sum'),
    avg_price = ('price',            'mean')
).reset_index()
category_sales = category_sales.sort_values('sales', ascending=False)
category_sales['sales_pct'] = (category_sales['sales'] / total_gmv * 100).round(1)

print("【指标 4 — 产品分类销售额 Top 10】")
top10_cat = category_sales.head(10).copy()
top10_cat['sales']   = top10_cat['sales'].apply(lambda x: f"R$ {x:,.0f}")
top10_cat['avg_price'] = top10_cat['avg_price'].apply(lambda x: f"R$ {x:,.2f}")
top10_cat.columns = ['品类', '销售额', '订单数', '商品数', '均价', '占比%']
print(top10_cat.to_string(index=False))

print(f"\n  Top 3 品类贡献: {category_sales['sales_pct'].head(3).sum():.1f}%")
print(f"  Top 10 品类贡献: {category_sales['sales_pct'].head(10).sum():.1f}%")

print("\n📊 **业务解读**：")
print("  品类集中度反映平台的品类结构健康度——")
print("  若 Top 3 占比过高（>50%），说明过度依赖少数品类，风险大；")
print("  若长尾品类贡献可观，说明平台生态多元化、抗风险能力强。")
print("  平均单价高的品类适合重点运营，提升利润空间。")

【指标 4 — 产品分类销售额 Top 10】
                   品类          销售额  订单数   商品数        均价  占比%
       bed_bath_table R$ 1,711,542 9417 16197  R$ 93.30  8.4
        health_beauty R$ 1,657,769 8836 12479 R$ 130.16  8.2
computers_accessories R$ 1,585,208 6689 11910 R$ 116.51  7.8
      furniture_decor R$ 1,429,656 6449 14779  R$ 87.56  7.0
        watches_gifts R$ 1,428,999 5624  7195 R$ 201.14  7.0
       sports_leisure R$ 1,392,027 7720 11216 R$ 114.34  6.9
           housewares R$ 1,094,669 5884 11142  R$ 90.79  5.4
                 auto   R$ 852,278 3897  5536 R$ 139.96  4.2
         garden_tools   R$ 838,186 3518  7399 R$ 111.63  4.1
           cool_stuff   R$ 779,524 3632  4369 R$ 167.36  3.8

  Top 3 品类贡献: 24.4%
  Top 10 品类贡献: 62.8%

📊 **业务解读**：
  品类集中度反映平台的品类结构健康度——
  若 Top 3 占比过高（>50%），说明过度依赖少数品类，风险大；
  若长尾品类贡献可观，说明平台生态多元化、抗风险能力强。
  平均单价高的品类适合重点运营，提升利润空间。


In [5]:
# ==== 5. 各州销售额排行 Top 10（客户地理位置）====
state_sales = df.groupby('customer_state').agg(
    sales     = ('order_total_amount', 'sum'),
    orders    = ('order_id',           'nunique'),
    customers = ('customer_id',        'nunique')
).reset_index()
state_sales = state_sales.sort_values('sales', ascending=False)
state_sales['sales_pct'] = (state_sales['sales'] / total_gmv * 100).round(1)
state_sales['avg_per_customer'] = (state_sales['sales'] / state_sales['customers']).round(2)

print("【指标 5 — 各州销售额 Top 10】")
top10_state = state_sales.head(10).copy()
top10_state['sales'] = top10_state['sales'].apply(lambda x: f"R$ {x:,.0f}")
top10_state.columns = ['州', '销售额', '订单数', '客户数', '占比%', '客均消费']
print(top10_state.to_string(index=False))

print(f"\n  覆盖州/地区数: {state_sales['customer_state'].nunique()}")

print("\n📊 **业务解读**：")
print("  区域销售分布反映市场渗透程度。")
print("  若集中在少数州，说明市场扩张空间大，可针对性增加广告投放和物流覆盖。")
print("  客均消费高的州是优质市场，值得深耕。")

【指标 5 — 各州销售额 Top 10】
 州          销售额   订单数   客户数  占比%   客均消费
SP R$ 7,596,842 41375 41375 37.4 183.61
RJ R$ 2,769,006 12762 12762 13.6 216.97
MG R$ 2,325,493 11544 11544 11.5 201.45
RS R$ 1,146,936  5432  5432  5.6 211.14
PR R$ 1,063,812  4998  4998  5.2 212.85
BA   R$ 797,277  3358  3358  3.9 237.43
SC   R$ 786,251  3612  3612  3.9 217.68
GO   R$ 513,766  2007  2007  2.5 255.99
DF   R$ 432,601  2125  2125  2.1 203.58
ES   R$ 405,781  2025  2025  2.0 200.39

  覆盖州/地区数: 27

📊 **业务解读**：
  区域销售分布反映市场渗透程度。
  若集中在少数州，说明市场扩张空间大，可针对性增加广告投放和物流覆盖。
  客均消费高的州是优质市场，值得深耕。


---
## 二、客户指标

In [6]:
# ==== 6. 总客户数、复购客户数、复购率 ====
total_customers = df['customer_id'].nunique()

# 从 customer_stats 聚合数据提取（每个 customer_id 一行）
cust_level = df.groupby('customer_id').agg(
    frequency = ('purchase_frequency', 'first')
).reset_index()
repeat_customers = (cust_level['frequency'] > 1).sum()
repurchase_rate  = repeat_customers / total_customers * 100

print("【指标 6 — 客户复购】")
print(f"  总客户数   : {total_customers:>10,}")
print(f"  复购客户数 : {repeat_customers:>10,}")
print(f"  复购率     : {repurchase_rate:>10.1f}%")

print("\n📊 **业务解读**：")
print("  复购率是衡量客户忠诚度的核心指标。")
print("  高复购率（>30%）说明产品/服务体验好、用户粘性强；")
print("  低复购率说明客户以一次性购买为主，需加强会员体系和复购激励。")

【指标 6 — 客户复购】
  总客户数   :     98,666
  复购客户数 :          0
  复购率     :        0.0%

📊 **业务解读**：
  复购率是衡量客户忠诚度的核心指标。
  高复购率（>30%）说明产品/服务体验好、用户粘性强；
  低复购率说明客户以一次性购买为主，需加强会员体系和复购激励。


In [7]:
# ==== 7. RFM 客户价值分层 ====
# 取每个客户的 RFM 指标（已在宽表中）
rfm = df.groupby('customer_id').agg(
    recency  = ('last_purchase_date',  'first'),   # 最近购买日期
    frequency = ('purchase_frequency', 'first'),    # 购买频次
    monetary  = ('total_spent',        'first')     # 消费总额
).reset_index()

# 计算参考日期（数据最大日期）
ref_date = df['purchase_time'].max()
rfm['recency_days'] = (ref_date - rfm['recency']).dt.days

# 按三分位数分层
def assign_score(series, ascending=True):
    """按三分位数分为 3 层：1=低, 2=中, 3=高"""
    q33 = series.quantile(0.33)
    q66 = series.quantile(0.66)
    def score(x):
        if x <= q33:
            return 3 if ascending else 1
        elif x <= q66:
            return 2
        else:
            return 1 if ascending else 3
    return series.apply(score)

rfm['R_score'] = assign_score(rfm['recency_days'], ascending=False)  # 越小越好
rfm['F_score'] = assign_score(rfm['frequency'],    ascending=True)   # 越大越好
rfm['M_score'] = assign_score(rfm['monetary'],     ascending=True)   # 越大越好

rfm['RFM_concat'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

def rfm_segment(concat_str):
    score_sum = sum(int(c) for c in concat_str)
    if score_sum >= 8:
        return '高价值客户'
    elif score_sum >= 5:
        return '中等价值客户'
    else:
        return '低价值客户'

rfm['segment'] = rfm['RFM_concat'].apply(rfm_segment)

seg_stats = rfm.groupby('segment').agg(
    count        = ('customer_id', 'count'),
    avg_recency  = ('recency_days', 'mean'),
    avg_freq     = ('frequency',    'mean'),
    avg_monetary = ('monetary',     'mean'),
    total_revenue = ('monetary',    'sum')
).reset_index()
seg_stats['pct'] = (seg_stats['count'] / total_customers * 100).round(1)
seg_stats['revenue_pct'] = (seg_stats['total_revenue'] / total_gmv * 100).round(1)

print("【指标 7 — RFM 客户价值分层】")
print(f"  参考日期: {ref_date.date()}\n")
print(seg_stats[['segment', 'count', 'pct', 'avg_recency', 'avg_freq', 'avg_monetary', 'revenue_pct']]
      .rename(columns={
          'segment': '分层', 'count': '客户数', 'pct': '占比%',
          'avg_recency': '平均距上次/天', 'avg_freq': '平均购买次数',
          'avg_monetary': '平均消费 R$', 'revenue_pct': '收入贡献%'
      }).to_string(index=False))

print("\n📊 **业务解读**：")
print("  RFM 模型是客户精细化运营的基础框架：")
print("  - 高价值客户：高频、高消费、近期活跃，是平台核心资产，需 VIP 专属服务")
print("  - 中等价值客户：有潜力向上转化，适合定向优惠券和个性化推荐")
print("  - 低价值客户：可做流失预警与唤醒召回")

【指标 7 — RFM 客户价值分层】
  参考日期: 2018-09-03

    分层   客户数  占比%    平均距上次/天  平均购买次数    平均消费 R$  收入贡献%
中等价值客户 65335 66.2 186.847601     1.0 207.957281   66.9
 高价值客户 33331 33.8 358.444331     1.0  67.704668   11.1

📊 **业务解读**：
  RFM 模型是客户精细化运营的基础框架：
  - 高价值客户：高频、高消费、近期活跃，是平台核心资产，需 VIP 专属服务
  - 中等价值客户：有潜力向上转化，适合定向优惠券和个性化推荐
  - 低价值客户：可做流失预警与唤醒召回


---
## 三、物流指标

In [8]:
# ==== 8. 平均物流天数、准时送达率 ====

# 只取已完成的订单
delivered = df[df['order_status_cn'] == '已完成'].copy()

avg_logistics = delivered['logistics_days'].mean()
median_logistics = delivered['logistics_days'].median()

# 准时送达 = 物流天数 <= 已完成的订单的物流天数中位数（模拟承诺送达）
# 实际业务中应有 estimated_delivery 字段，这里用中位数作为基线
# 或者：物流天数在 0~14 天内的视为正常配送
on_time_base = 14  # 14 天内送达视为准时（Olist 巴西平均时效）
on_time_count = (delivered['logistics_days'] <= on_time_base).sum()
on_time_rate  = on_time_count / len(delivered) * 100

# 分段统计
logistics_bins = [0, 3, 7, 14, 21, 30, 100]
logistics_labels = ['0-3天', '4-7天', '8-14天', '15-21天', '22-30天', '30天+']
delivered['logistics_range'] = pd.cut(delivered['logistics_days'], bins=logistics_bins, labels=logistics_labels)
logistics_dist = delivered['logistics_range'].value_counts().sort_index()
logistics_dist_pct = (logistics_dist / len(delivered) * 100).round(1)

print("【指标 8 — 物流时效】")
print(f"  已完成订单数   : {len(delivered):,}")
print(f"  平均物流天数   : {avg_logistics:.1f} 天")
print(f"  物流天数中位数 : {median_logistics:.0f} 天")
print(f"  准时送达率     : {on_time_rate:.1f}% (≤{on_time_base}天)")
print(f"\n  物流天数分布：")
for label, pct in logistics_dist_pct.items():
    print(f"    {label}: {pct:>5.1f}%")

print("\n📊 **业务解读**：")
print("  物流时效直接影响客户满意度和复购意愿。")
print("  若平均物流天数过高或准时率偏低，需优化仓储网络布局或更换物流服务商。")
print("  配送集中在 7-14 天说明可探索次日达/隔日达等高时效服务作为差异化竞争力。")

【指标 8 — 物流时效】
  已完成订单数   : 110,197
  平均物流天数   : 12.0 天
  物流天数中位数 : 10 天
  准时送达率     : 73.0% (≤14天)

  物流天数分布：
    0-3天:   9.0%
    4-7天:  26.1%
    8-14天:  37.9%
    15-21天:  15.9%
    22-30天:   7.0%
    30天+:   4.1%

📊 **业务解读**：
  物流时效直接影响客户满意度和复购意愿。
  若平均物流天数过高或准时率偏低，需优化仓储网络布局或更换物流服务商。
  配送集中在 7-14 天说明可探索次日达/隔日达等高时效服务作为差异化竞争力。


In [9]:
# ==== 9. 各产品分类的平均物流天数排行 ====
cat_logistics = delivered.groupby('product_category_name_english').agg(
    avg_logistics = ('logistics_days', 'mean'),
    order_count   = ('order_id',       'nunique')
).reset_index()
# 至少 50 单才有统计意义
cat_logistics = cat_logistics[cat_logistics['order_count'] >= 50]
cat_logistics = cat_logistics.sort_values('avg_logistics')
cat_logistics['avg_logistics'] = cat_logistics['avg_logistics'].round(1)

print("【指标 9 — 各品类平均物流天数 Top & Bottom 10】")
print("\n--- 最快送达 Top 10 ---")
print(cat_logistics.head(10).rename(columns={
    'product_category_name_english': '品类',
    'avg_logistics': '平均物流天数',
    'order_count': '订单数'
}).to_string(index=False))

print("\n--- 最慢送达 Top 10 ---")
print(cat_logistics.tail(10).sort_values('avg_logistics', ascending=False).rename(columns={
    'product_category_name_english': '品类',
    'avg_logistics': '平均物流天数',
    'order_count': '订单数'
}).to_string(index=False))

print("\n📊 **业务解读**：")
print("  不同品类的物流时效差异反映供应链能力：")
print("  - 快送品类（如小型配件、数字商品）可作为引流主打")
print("  - 慢送品类（如大家具、大型家电）需在商品页提前告知时效，管理用户预期")

【指标 9 — 各品类平均物流天数 Top & Bottom 10】

--- 最快送达 Top 10 ---
                                   品类  平均物流天数  订单数
                       books_imported     7.7   50
                                 food     9.1  441
            construction_tools_lights     9.2  242
small_appliances_home_oven_and_coffee     9.4   72
               signaling_and_security    10.0  138
                               drinks    10.0  287
                           cine_photo    10.1   63
                  luggage_accessories    10.2 1019
                      books_technical    10.2  256
      construction_tools_construction    10.3  736

--- 最慢送达 Top 10 ---
                     品类  平均物流天数  订单数
       office_furniture    20.4 1254
     christmas_supplies    15.3  125
          fashion_shoes    14.9  235
      home_appliances_2    13.4  227
fashion_underwear_beach    13.3  117
  furniture_living_room    13.3  414
           garden_tools    13.2 3448
              computers    13.1  177
         consoles_games    13

---
## 四、支付指标

In [10]:
# ==== 10. 各支付方式使用占比 ====
pay_stats = df.groupby('payment_type').agg(
    count   = ('order_id',       'nunique'),
    total   = ('payment_value',  'sum'),
    avg_pay = ('payment_value',  'mean'),
    avg_installments = ('payment_installments', 'mean')
).reset_index()
pay_stats['count_pct'] = (pay_stats['count'] / total_orders * 100).round(1)
pay_stats['gmv_pct']   = (pay_stats['total'] / total_gmv * 100).round(1)
pay_stats = pay_stats.sort_values('count', ascending=False)

print("【指标 10 — 支付方式占比】")
print(pay_stats.rename(columns={
    'payment_type': '支付方式',
    'count': '订单数',
    'count_pct': '订单占比%',
    'gmv_pct': 'GMV占比%',
    'total': '交易总额',
    'avg_pay': '笔均金额',
    'avg_installments': '平均分期数'
}).to_string(index=False))

print("\n📊 **业务解读**：")
print("  支付方式分布反映用户支付偏好和平台金融能力：")
print("  - 信用卡占比高说明用户消费能力较强，适合推高价商品和分期免息活动")
print("  - 借记卡/现金占比高说明价格敏感型用户多，可推低价爆品引流")
print("  - 分期数高低反映用户对大额消费的接受度，可针对性设计分期免息促销")

【指标 10 — 支付方式占比】
       支付方式   订单数        交易总额       笔均金额    平均分期数  订单占比%  GMV占比%
credit_card 75962 15526968.76 179.712367 3.628283   77.0    76.5
     boleto 19613  4059605.20 177.538931 1.000000   19.9    20.0
    voucher  1540   150241.00  90.725242 1.000000    1.6     0.7
 debit_card  1471   246127.30 150.352657 1.000000    1.5     1.2

📊 **业务解读**：
  支付方式分布反映用户支付偏好和平台金融能力：
  - 信用卡占比高说明用户消费能力较强，适合推高价商品和分期免息活动
  - 借记卡/现金占比高说明价格敏感型用户多，可推低价爆品引流
  - 分期数高低反映用户对大额消费的接受度，可针对性设计分期免息促销


---
## 汇总仪表盘

In [11]:
# ==== 最终汇总 ====
print("=" * 60)
print("           Olist 电商 KPI 仪表盘")
print("=" * 60)
print(f"\n📦 销售")
print(f"  GMV          : R$ {total_gmv:>15,.0f}")
print(f"  订单量       : {total_orders:>15,}")
print(f"  平均客单价   : R$ {avg_order_value:>12,.2f}")
print(f"  平均商品数   : {avg_item_count:>15.1f} 件")
print(f"  Top1 品类    : {category_sales.iloc[0]['product_category_name_english']}")
print(f"  Top1 州      : {state_sales.iloc[0]['customer_state']}")

print(f"\n👤 客户")
print(f"  总客户数     : {total_customers:>15,}")
print(f"  复购率       : {repurchase_rate:>15.1f}%")
high_val = seg_stats[seg_stats['segment'] == '高价值客户']
if len(high_val) > 0:
    print(f"  高价值客户   : {high_val.iloc[0]['count']:>15,} ({high_val.iloc[0]['pct']}%)")

print(f"\n🚚 物流")
print(f"  平均物流天数 : {avg_logistics:>15.1f} 天")
print(f"  准时送达率   : {on_time_rate:>15.1f}%")

print(f"\n💳 支付")
top_pay = pay_stats.iloc[0]
print(f"  最常用支付   : {top_pay['payment_type']} ({top_pay['count_pct']}%)")

print("\n" + "=" * 60)

# 写入 SQLite 供后续使用
kpi_conn = sqlite3.connect(DB_PATH)
monthly.to_sql('kpi_monthly_trend', kpi_conn, if_exists='replace', index=False)
category_sales.to_sql('kpi_category_sales', kpi_conn, if_exists='replace', index=False)
state_sales.to_sql('kpi_state_sales', kpi_conn, if_exists='replace', index=False)
seg_stats.to_sql('kpi_rfm_segments', kpi_conn, if_exists='replace', index=False)
pay_stats.to_sql('kpi_payment_stats', kpi_conn, if_exists='replace', index=False)
cat_logistics.to_sql('kpi_category_logistics', kpi_conn, if_exists='replace', index=False)
kpi_conn.close()

print("[DONE] 所有指标表已写入 olist.db")
conn.close()

           Olist 电商 KPI 仪表盘

📦 销售
  GMV          : R$      20,304,557
  订单量       :          98,666
  平均客单价   : R$       160.58
  平均商品数   :             1.1 件
  Top1 品类    : bed_bath_table
  Top1 州      : SP

👤 客户
  总客户数     :          98,666
  复购率       :             0.0%
  高价值客户   :          33,331 (33.8%)

🚚 物流
  平均物流天数 :            12.0 天
  准时送达率   :            73.0%

💳 支付
  最常用支付   : credit_card (77.0%)

[DONE] 所有指标表已写入 olist.db
